# GitLab Authentifizierung

Die GitLab-Authentifizierung ermöglicht die Anmeldung an Backstage über eine GitLab OAuth Application.

Das Auth-Modul ist unabhängig von der GitLab Catalog Discovery. Die Catalog Discovery importiert Repositories, während der Auth Provider Benutzer authentifiziert und einer Backstage `User` Entity zuordnet.


## GitLab Auth Provider installieren

Das GitLab Auth Provider Modul wird im Backstage-Backend installiert.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
yarn --cwd packages/backend add @backstage/plugin-auth-backend-module-gitlab-provider

## Backend-Modul registrieren

Das Auth-Modul wird im Backstage-Backend registriert, damit die GitLab OAuth Endpunkte beim Start bereitgestellt werden.

Dazu patchen wir die Datei [packages/backend/src/index.ts](../../mybackstage/packages/backend/src/index.ts)


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

grep -q "plugin-auth-backend-module-gitlab-provider" packages/backend/src/index.ts \
|| sed -i "/backend.start();/i backend.add(import('@backstage/plugin-auth-backend-module-gitlab-provider'));" packages/backend/src/index.ts

# Kontrolle
yarn why @backstage/plugin-auth-backend-module-gitlab-provider

## GitLab OAuth Application erstellen

In GitLab:

* Profilbild öffnen
* **Edit profile**
* **Access → Applications → Add new application**
* Namen setzen, beispielsweise `Backstage GitLab Auth`
* Als Redirect URI die unten ausgegebene URL eintragen
* Scope `read_user` auswählen
* Application erstellen
* `Application ID` und `Secret` kopieren
* Werte in [env-platen.py](../../data/env-platen.py) als `AUTH_GITLAB_CLIENT_ID` und `AUTH_GITLAB_CLIENT_SECRET` eintragen

Für die reine Anmeldung reicht der Scope `read_user`. Die Redirect URI darf nach `frame` keinen abschliessenden Slash enthalten.


In [ ]:
%%bash
export BACKSTAGE_HOSTNAME=localhost

echo "GitLab Redirect URI:"
echo "http://${BACKSTAGE_HOSTNAME}:7007/api/auth/gitlab/handler/frame"


## Umgebungsvariablen prüfen

Die OAuth-Zugangsdaten werden aus `env-platen.py` geladen. Das Script zeigt nur, ob die Variablen gesetzt sind, und gibt keine Secrets aus.


In [ ]:
%%bash
set -a
source ~/data/env-platen.py
set +a

test -n "${AUTH_GITLAB_CLIENT_ID}" \
  && echo "AUTH_GITLAB_CLIENT_ID ist gesetzt" \
  || echo "AUTH_GITLAB_CLIENT_ID fehlt"

test -n "${AUTH_GITLAB_CLIENT_SECRET}" \
  && echo "AUTH_GITLAB_CLIENT_SECRET ist gesetzt" \
  || echo "AUTH_GITLAB_CLIENT_SECRET fehlt"


## GitLab Auth Provider konfigurieren

Der GitLab Provider wird in einer separaten Backstage-Konfigurationsdatei eingerichtet.

Der Resolver `usernameMatchingUserEntityName` ordnet den GitLab-Benutzernamen einer Backstage `User` Entity mit demselben `metadata.name` zu.

Beispiel: Der GitLab-Benutzer `marcel` benötigt im Backstage Catalog eine `User` Entity mit `metadata.name: marcel`.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

cat > app-config.gitlab-auth.yaml <<'EOF'
auth:
  environment: development
  providers:
    gitlab:
      development:
        clientId: ${AUTH_GITLAB_CLIENT_ID}
        clientSecret: ${AUTH_GITLAB_CLIENT_SECRET}
        signIn:
          resolvers:
            - resolver: usernameMatchingUserEntityName
EOF


## GitLab Login im Frontend konfigurieren

Die neue Backstage Frontend-Architektur verwendet eine `SignInPageBlueprint` Extension.

Das folgende Script erstellt die Extension in `packages/app/src/extensions/gitlabSignInPage.tsx`. Die Extension muss danach in der bestehenden `features`-Liste von `packages/app/src/App.tsx` importiert und ergänzt werden.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

mkdir -p packages/app/src/extensions

cat > packages/app/src/extensions/gitlabSignInPage.tsx <<'EOF'
import { SignInPage } from '@backstage/core-components';
import { gitlabAuthApiRef } from '@backstage/core-plugin-api';
import { SignInPageBlueprint } from '@backstage/plugin-app-react';

export const gitlabSignInPage = SignInPageBlueprint.make({
  params: {
    loader: async () => props => (
      <SignInPage
        {...props}
        provider={{
          id: 'gitlab-auth-provider',
          title: 'GitLab',
          message: 'Mit GitLab anmelden',
          apiRef: gitlabAuthApiRef,
        }}
      />
    ),
  },
});
EOF

echo "Extension erstellt:"
echo "packages/app/src/extensions/gitlabSignInPage.tsx"


## Frontend Extension registrieren

Das Script ergänzt den Import und fügt `gitlabSignInPage` am Anfang der vorhandenen `features`-Liste ein.

Vor der Änderung wird eine Sicherung von `App.tsx` erstellt. Falls die erwartete `features`-Liste nicht gefunden wird, bricht das Script ohne Änderung ab.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

python3 - <<'PY'
from pathlib import Path
import shutil

path = Path("packages/app/src/App.tsx")
backup = path.with_suffix(".tsx.gitlab-auth.bak")
text = path.read_text(encoding="utf-8")

import_line = "import { gitlabSignInPage } from './extensions/gitlabSignInPage';"

if import_line not in text:
    lines = text.splitlines()
    last_import = max(
        (index for index, line in enumerate(lines) if line.startswith("import ")),
        default=-1,
    )
    lines.insert(last_import + 1, import_line)
    text = "\n".join(lines) + "\n"

if "gitlabSignInPage," not in text:
    marker = "features: ["
    if marker not in text:
        raise SystemExit(
            "Keine features-Liste gefunden. App.tsx wurde nicht verändert."
        )
    text = text.replace(marker, marker + "\n    gitlabSignInPage,", 1)

if not backup.exists():
    shutil.copy2(path, backup)

path.write_text(text, encoding="utf-8")
print(f"App.tsx aktualisiert. Sicherung: {backup}")
PY

grep -n "gitlabSignInPage" packages/app/src/App.tsx


## Backstage User Entity prüfen

Der gewählte Resolver benötigt eine passende `User` Entity im Software Catalog.

Die bestehende User-Konfiguration kann mit dem folgenden Befehl gesucht werden.


In [ ]:
%%bash
cd ~/mybackstage/

grep -R --line-number --include='*.yaml' --include='*.yml' \
  -E '^kind:[[:space:]]*User|^[[:space:]]*name:[[:space:]]*marcel' \
  examples catalog 2>/dev/null || true


## Konfiguration prüfen

Mit `yarn backstage-cli config:print` wird die zusammengeführte und aufgelöste Backstage-Konfiguration ausgegeben, ohne die Anwendung zu starten.


In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage GitLab Auth"
export BACKSTAGE_PORT="3001"

source ~/.nvm/nvm.sh
cd ~/mybackstage

yarn backstage-cli config:print \
  --config ~/mybackstage/app-config.yaml \
  --config ~/mybackstage/app-config.test.yaml \
  --config ~/mybackstage/app-config.gitlab.yaml \
  --config ~/mybackstage/app-config.gitlab-auth.yaml


## Backstage starten

Der Start-Endpunkt (Backend) des GitLab Providers muss mit einem HTTP Redirect antworten.


In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage GitLab Auth"
export BACKSTAGE_PORT="3001"

echo "Frontend: http://$(cat ~/data/server-ip):${BACKSTAGE_PORT}"
echo "Backend:  http://localhost:7007/api/auth/gitlab/start?env=development"

source ~/.nvm/nvm.sh
cd ~/mybackstage

yarn start \
  --config ~/mybackstage/app-config.yaml \
  --config ~/mybackstage/app-config.test.yaml \
  --config ~/mybackstage/app-config.gitlab.yaml \
  --config ~/mybackstage/app-config.gitlab-auth.yaml \
  2>&1 | tee /tmp/backstage-gitlab-auth.log
